# Trajectory optimization for a 2 link planar manipulator

Import statements

In [108]:
import numpy as np
import math
import matplotlib.pyplot as plt
from casadi import *
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

## Class Definition
Build a class for storing all manipulator properties and EoMs

In [109]:
## Class for Manipulator Dynamics
class Manipulator:
    """Class for a 2 link Manipulator optimization problem."""

    def __init__(self, masses, lengths, init_xy, final_xy,
                 objective=0, steps=1000, max_iter=100,
                 theta_lims=(0,math.pi/2,-math.pi,math.pi),
                 max_VAT=(2,1,100), max_time=10000):

        # Initialize optimizer object
        self.opti = Opti()
    
        # Manipulator parameters
        self.M1         = masses[0]
        self.M2         = masses[1]
        self.L1         = lengths[0]
        self.L2         = lengths[1]
        self.init_xy    = init_xy
        self.final_xy   = final_xy
        self.theta_lims = theta_lims
        self.max_VAT    = max_VAT
        
        # Define simulation parameters
        self.N          = steps
        self.g          = 9.81
        self.max_iter   = max_iter
        self.objective  = objective
        self.max_time   = max_time

        # Define arrays for system matrices
        self.c     = MX.sym('c'    , 2,    self.N)
        self.G     = MX.sym("G"    , 2,    self.N)
        self.M_inv = MX.sym('M_inv', 2, 2, self.N) #list of MX

        # Define arrays for state trajectory
        self.X = self.opti.variable(6,self.N)
        self.theta1       = self.X[0,:]
        self.theta2       = self.X[1,:]
        self.theta1_dot   = self.X[2,:]
        self.theta2_dot   = self.X[3,:]
        self.theta1_d_dot = self.X[4,:]
        self.theta2_d_dot = self.X[5,:]

        # Time variables
        self.T = self.opti.variable()
        self.dt = self.T / self.N
        
        # Define decision variables. N-1 because X_N is decided by tau_N-1
        self.tau = self.opti.variable(2, self.N-1)

    #-----INVERSE KINEMATICS--------------------------------------------
    
    def inverse_kinematics_2link(self, target_end_effector_pos):
        """Determine theta angles given xy coordinates."""

        # Extract manipulator parameters
        L1 = self.L1
        L2 = self.L2
        theta1_min, theta1_max, theta2_min, theta2_max = self.theta_lims
        x, y = target_end_effector_pos

        # Check if y-coordinate is non-negative
        if y < 0:
            return "Error: Y-coordinate must be greater than or equal to 0."

        # Determine endpoint total angle to ground
        d2 = x**2 + y**2
        d = np.sqrt(d2)
        base_angle = np.arctan2(y, x)

        # Cosine of the first angle in a triangle defined by L1 and L2
        C = (L1**2 + d2 - L2**2) / (2 * L1 * d)
        c = np.arccos(C)

        # Check if the target is reachable
        if not -1 <= C <= 1:
            return "Error: Target is outside the reachable workspace."

        # Cosine of the second angle
        B = (L1**2 + L2**2 - d2) / (2 * L1 * L2)
        b = np.arccos(B)

        # Calculate joint angles for elbow up configuration
        theta1_elbow_up = base_angle + c
        theta2_elbow_up = b - np.pi

        # Check if elbow up joint angles are within specified limits
        if (theta1_min <= theta1_elbow_up <= theta1_max) and (theta2_min <= theta2_elbow_up <= theta2_max):
            theta1 = theta1_elbow_up
            theta2 = theta2_elbow_up
            return (theta1, theta2)

        # Calculate joint angles for elbow down configuration
        theta1_elbow_down = base_angle - c
        theta2_elbow_down = np.pi - b

        # Check if elbow down joint angles are within specified limits
        if (theta1_min <= theta1_elbow_down <= theta1_max) and (theta2_min <= theta2_elbow_down <= theta2_max):
            theta1 = theta1_elbow_down
            theta2 = theta2_elbow_down
            return (theta1, theta2)

        return "Error: No valid joint angles found within the specified limits."

    #-----DYNAMICS -----------------------------------------------------

    def make_M_inv(self):
        """Sets up the array of mass matrices."""

        # Matrix elements
        a   = (self.M1 + self.M2) * self.L1**2
        d   = self.M2 * self.L2**2

        for i in range(self.N):
            b_c = self.M2 * self.L1 * self.L2 * cos(self.theta1[i] - self.theta2[i])

            # M_inv matrix, invertible iff masses and lengths are non-zero
            determinant = a * d - b_c ** 2

            self.M_inv[i][0, 0] = d     / determinant
            self.M_inv[i][0, 1] = -b_c  / determinant
            self.M_inv[i][1, 0] = -b_c  / determinant
            self.M_inv[i][1, 1] = a     / determinant
        return

    def make_c(self):
        """Sets up the array of coriolis matrices."""

        # Fill in c matrix
        for i in range(self.N):
            common = self.M2 * self.L1 * self.L2 * sin(self.theta1[i]-self.theta2[i])
            self.c[0, i] =  common * self.theta2_dot[i]**2
            self.c[1, i] = -common * self.theta1_dot[i]**2
        return

    def make_G(self):
        """Sets up the array of gravity matrices."""

        # Fill in G matrix
        for i in range(self.N):
            self.G[0, i] = (self.M1+self.M2) * self.g * self.L1 * cos(self.theta1[i])
            self.G[1, i] = self.M2 * self.g * self.L2 * cos(self.theta2[i])
        return

    def compute_accel(self):
        """Compute the acceleration given the torque and forces."""

        # Determine the current acceleration frame by frame
        for k in range(0, self.N-1):
            B = self.tau[:, k] - self.c[:, k] - self.G[:, k]
            self.X[4:6, k] = self.M_inv[k] @ B
        return
    
    #-----CONSTRAINTS---------------------------------------------------

    def set_boundary_conditions(self):
        """Apply constraints to optimizer of boundary states."""

        # Define starting angles
        self.opti.subject_to(self.theta1[0] == self.init_theta[0])
        self.opti.subject_to(self.theta2[0] == self.init_theta[1])
        self.opti.subject_to(self.theta1[-1]== self.final_theta[0])
        self.opti.subject_to(self.theta2[-1]== self.final_theta[1])

        # Zero velocity at start and end
        self.opti.subject_to(self.theta1_dot[0]  == 0)
        self.opti.subject_to(self.theta2_dot[0]  == 0)
        self.opti.subject_to(self.theta1_dot[-1] == 0)
        self.opti.subject_to(self.theta2_dot[-1] == 0)

        # Zero acceleration at start and end
        self.opti.subject_to(self.theta1_d_dot[0]  == 0)
        self.opti.subject_to(self.theta2_d_dot[0]  == 0)
        self.opti.subject_to(self.theta1_d_dot[-1] == 0)
        self.opti.subject_to(self.theta2_d_dot[-1] == 0)
        return

    def build_time_constraints(self):
        """Apply min and max time constraints."""

        # Time has to be non-negative and finite
        self.opti.subject_to(self.T >= 0)
        self.opti.subject_to(self.T < self.max_time)
        return
    
    def build_equality_constraints(self):
        """Continuity for angle position and angular velocities."""

        # Ensure that the next angle and vel are based on current vel and acc, respectively
        for k in range(self.N-1):
            self.opti.subject_to(self.theta1[k+1] == self.theta1[k] + self.dt * self.theta1_dot[k])
            self.opti.subject_to(self.theta2[k+1] == self.theta2[k] + self.dt * self.theta2_dot[k])
            self.opti.subject_to(self.theta1_dot[k+1] == self.theta1_dot[k] + self.dt * self.theta1_d_dot[k])
            self.opti.subject_to(self.theta2_dot[k+1] == self.theta2_dot[k] + self.dt * self.theta2_d_dot[k])
        return

    def theta_and_tau_constraints(self):
        """Apply constraints for the permissable state and torque values."""

        # Extract limits
        theta1_min, theta1_max, theta2_min, theta2_max = self.theta_lims
        max_abs_ang_vel, max_abs_ang_acc, max_abs_tau = self.max_VAT # Absolute ang. vel, acc, torque

        # Within angle and tau limits
        self.opti.subject_to(self.opti.bounded(theta1_min,self.theta1,theta1_max))
        self.opti.subject_to(self.opti.bounded(theta2_min,self.theta2,theta2_max))
        self.opti.subject_to(self.opti.bounded(-max_abs_ang_vel,self.theta1_dot,max_abs_ang_vel))
        self.opti.subject_to(self.opti.bounded(-max_abs_ang_vel,self.theta2_dot,max_abs_ang_vel))
        self.opti.subject_to(self.opti.bounded(-max_abs_ang_acc,self.theta1_d_dot,max_abs_ang_acc))
        self.opti.subject_to(self.opti.bounded(-max_abs_ang_acc,self.theta2_d_dot,max_abs_ang_acc))
        self.opti.subject_to(self.opti.bounded(-max_abs_tau,self.tau,max_abs_tau))
        return
    
    #-----OBJECTIVE FUNCTION--------------------------------------------
    
    def build_objectives(self):
        """Define objective function to minimize."""

        # Minimize input energy, P = torque * omega, E = integral(P dt)
        if self.objective == 0:
            total_energy = 0
            for i in range(self.N - 1):
                power1 = self.tau[0, i] * self.theta1_dot[i]
                power2 = self.tau[1, i] * self.theta2_dot[i]
                total_energy += self.dt * (power1 + power2)
            self.opti.minimize(total_energy)
        
         # Alternative objective function to be as fast as feasible
        elif self.objective == 1:  
            self.opti.minimize(self.T) 
        
        # Minimize input magnitude
        else:
            sum_tau = 0
            for i in range(self.N - 1):
                sum_tau += (fabs(self.tau[0, i]) + fabs(self.tau[1, i]))
            self.opti.minimize(sum_tau)
        return
    
    #-----WARM-START SOLVER---------------------------------------------
        
    def initialize_solver(self):
        """Warm-start values for solver."""
        
        # Starting at guess of Time == 10 seconds, arbitrary
        self.opti.set_initial(self.T, 10)
        return
    
    #----MAIN SOLVER----------------------------------------------------

    def solve_optimizer(self):
        """Main solver method.
        Full Optimization process set up in 7 stages:\n
            1. Perform inverse kinematics to find the boundary angles.\n
            2. Choose a solver for the optimizer object.\n
            3. Establish the dynamics of the system.\n
            4. Add constraints relating to boundary conditions, time, max inputs and outputs, and ensure state continuity.\n
            5. Define the objective function that is to be minimized.\n
            6. Initialize values for the optimizer.\n
            7. Solve the system
        """
        
        # Inverse kinematics
        self.init_theta = self.inverse_kinematics_2link(self.init_xy)
        self.final_theta = self.inverse_kinematics_2link(self.final_xy)

        # Set solver. ipopt is most common, but more options available
        p_opts = {"record_time":True}
        s_opts = {"max_iter": self.max_iter, "acceptable_tol": 1e-6}
        self.opti.solver('ipopt', p_opts, s_opts) 

        # Dynamics equations and constraints
        self.make_M_inv()
        self.make_c()
        self.make_G()
        self.compute_accel()
        
        # Simulation constraints
        self.set_boundary_conditions()
        self.build_time_constraints()
        self.build_equality_constraints()
        self.theta_and_tau_constraints()

        # Objective Function
        self.build_objectives()

        # Initial Guess to warm-start optimizer
        self.initialize_solver()

        # Solve optimization problem
        sol = self.opti.solve()
        return sol    

## Plot Methods in Class

In [110]:
def plot_thetas(self, sol):
    """Plot the States over Time."""

    # Extract values
    total_time = sol.value(self.T)
    num_samples = self.N
    theta1 = sol.value(self.theta1)
    theta2 = sol.value(self.theta2)
    theta1_dot = sol.value(self.theta1_dot)
    theta2_dot = sol.value(self.theta2_dot)
    theta1_d_dot = sol.value(self.theta1_d_dot)
    theta2_d_dot = sol.value(self.theta2_d_dot)

    # Create time array
    time = np.linspace(0, total_time, num_samples, endpoint=True)

    # Create subplots for the first three pairs of data
    fig, axs = plt.subplots(3, 1, figsize=(10, 18))

    # Plot theta1 vs. theta2
    axs[0].plot(time, theta1, label=r'$\theta_1$', color='blue')
    axs[0].plot(time, theta2, label=r'$\theta_2$', color='orange')
    axs[0].set_xlabel('Time (s)')
    axs[0].set_ylabel('Value')
    axs[0].set_title(r'$\theta_1$ vs. $\theta_2$')
    axs[0].legend()
    axs[0].grid(True)

    # Plot theta1_dot vs. theta2_dot
    axs[1].plot(time, theta1_dot, label=r'$\dot{\theta}_1$', color='green')
    axs[1].plot(time, theta2_dot, label=r'$\dot{\theta}_2$', color='red')
    axs[1].set_xlabel('Time (s)')
    axs[1].set_ylabel('Value')
    axs[1].set_title(r'$\dot{\theta}_1$ vs. $\dot{\theta}_2$')
    axs[1].legend()
    axs[1].grid(True)

    # Plot theta1_d_dot vs. theta2_d_dot
    axs[2].plot(time, theta1_d_dot, label=r'$\ddot{\theta}_1$', color='purple')
    axs[2].plot(time, theta2_d_dot, label=r'$\ddot{\theta}_2$', color='cyan')
    axs[2].set_xlabel('Time (s)')
    axs[2].set_ylabel('Value')
    axs[2].set_title(r'$\ddot{\theta}_1$ vs. $\ddot{\theta}_2$')
    axs[2].legend()
    axs[2].grid(True)

    plt.tight_layout()
    plt.show()
    return

def plot_taus(self, sol):
    """Plot the Torques over Time."""

    # Extract values
    num_samples = self.N
    total_time = sol.value(self.T)
    tau1 = sol.value(self.tau[0, :])
    tau2 = sol.value(self.tau[1, :])

    # Create time array
    time = np.linspace(0, total_time, num_samples, endpoint=True)

    # Plot tau1 vs. tau2
    plt.figure(figsize=(10, 6))
    plt.plot(time, tau1, label=r'$\tau_1$', color='magenta')
    plt.plot(time, tau2, label=r'$\tau_2$', color='yellow')
    plt.xlabel('Time (s)')
    plt.ylabel('Value')
    plt.title(r'$\tau_1$ vs. $\tau_2$')
    plt.legend()
    plt.grid(True)
    plt.show()
    return

def animate_trajectory(self, sol):
    """Animate the trajectory of the manipulator."""

    # Extract values
    L1 = self.L1
    L2 = self.L2
    theta1_values = sol.value(self.theta1)
    theta2_values = sol.value(self.theta2)

    # Create subplot for data
    fig, ax = plt.subplots(figsize=(8, 6))

    # Initialize empty plots for links and joints
    link1, = ax.plot([], [], 'blue', linewidth=2, label='Link 1')
    link2, = ax.plot([], [], 'orange', linewidth=2, label='Link 2')
    base, = ax.plot([], [], 'ko')
    joint1, = ax.plot([], [], 'ko')
    end_effector, = ax.plot([], [], 'ko')

    # Plot parameters
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('2-Link Manipulator Optimization Animation')
    ax.set_aspect('equal')
    ax.grid(True)
    ax.legend()
    ax.set_xlim(-L1 - L2, L1 + L2)
    ax.set_ylim(-L1 - L2, L1 + L2)

    def update(frame):
        """Update function for the animation."""

        # Calculate joint positions for the current frame
        theta1 = theta1_values[frame]
        theta2 = theta2_values[frame]
        x1 = L1 * np.cos(theta1)
        y1 = L1 * np.sin(theta1)
        x2 = x1 + L2 * np.cos(theta1 + theta2)
        y2 = y1 + L2 * np.sin(theta1 + theta2)

        # Update the plots
        link1.set_data([0, x1], [0, y1])
        link2.set_data([x1, x2], [y1, y2])
        base.set_data([0], [0])
        joint1.set_data([x1], [y1])
        end_effector.set_data([x2], [y2])

        return link1, link2, base, joint1, end_effector

    # Create the animation
    anim = FuncAnimation(fig, update, frames=len(theta1_values), interval=50, blit=True)
    plt.close(fig)  
    return anim

# Add functions as methods to class
Manipulator.plot_thetas = plot_thetas
Manipulator.plot_taus = plot_taus
Manipulator.animate_trajectory = animate_trajectory
    

## Initialize Test Manipulator

In [111]:
# Manipulator input parameters
init_xy = (2.5, 1)
final_xy = (1.5, 0.25)
masses = (6, 3)
lengths = (2, 1)
objective = 0  # 0: Input energy, 1: time, 2: input magnitude

# Initialize Manipulator object and solve
sample_manipulator = Manipulator(masses, lengths, init_xy, final_xy, objective)
optimizer_solution = sample_manipulator.solve_optimizer()

# Extract results and statistics
result = optimizer_solution.stats()['return_status']
iter_count = optimizer_solution.stats()['iter_count']
solution_time = optimizer_solution.stats()['time_cpu_solver']
optimal_cost = optimizer_solution.value


# Print the information
print("Result:", result)
print("Iterations:", iter_count)
print("Solution Time:", solution_time)
print("Optimal Solution Cost:", optimal_cost)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:    15996
Number of nonzeros in inequality constraint Jacobian.:     8000
Number of nonzeros in Lagrangian Hessian.............:     7992

Total number of variables............................:     7999
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:     4008
Total number of inequality constraints...............:     8000
        inequality constraints with only lower bounds:        1
   inequality constraints with lower and upper bounds:     7998
        inequality constraints with only upper bounds:        1

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 2.31e+00 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

RuntimeError: Error in Opti::solve [OptiNode] at .../casadi/core/optistack.cpp:157:
.../casadi/core/optistack_internal.cpp:996: Assertion "return_success(accept_limit)" failed:
Solver failed. You may use opti.debug.value to investigate the latest values of variables. return_status is 'Maximum_Iterations_Exceeded'

Plots of thetas and their respective derivatives

In [ ]:
sample_manipulator.plot_thetas(optimizer_solution)

Plots of the applied torques

In [ ]:
sample_manipulator.plot_taus(optimizer_solution)

Animation of the manipulator trajectory

In [ ]:
anim = sample_manipulator.animate_trajectory(optimizer_solution)
HTML(anim.to_html5_video())